In [ ]:
# Importing helpful packages 

import tensorflow as tf
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from sklearn.metrics import confusion_matrix
from mlxtend.plotting import plot_confusion_matrix
import seaborn as sns
import keras
from keras import models
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from keras.optimizers import RMSprop,Adam
from keras.utils import to_categorical
from keras.preprocessing.image import ImageDataGenerator

In [ ]:
data = pd.read_csv('C:\\Users\\LapStore\\Downloads\\Tools\\icml_face_data.csv')


# 1) Understand the dataset and Clean


In [ ]:
# Display basic information about the DataFrame
print(data.info())

In [ ]:
# Display descriptive statistics
print(data.describe())

In [ ]:
# Display the first few rows of the DataFrame
print(data.head(5))

In [ ]:
# 2. Handling missing values
# Check for missing values in each column
print(data.isnull().sum())

In [ ]:
# 3. Handling duplicate values
# Check for duplicate rows
duplicate_rows = data.duplicated()

# Display the duplicate rows
print("Duplicate Rows except first occurrence:")
print(data[duplicate_rows])

In [ ]:
# Remove duplicate rows
remove_duplicates = data.drop_duplicates()

# Display the DataFrame without duplicates
print("Data after removing duplicates:")
print(remove_duplicates)

In [ ]:


# Creating a few useful functions

def plot_all_emotions():
    fig, axs = plt.subplots(1, 7, figsize=(30, 12))
    fig.subplots_adjust(hspace = .2, wspace=.2)
    axs = axs.ravel()
    for i in range(7):
        idx = data[data['emotion']==i].index[i]
        axs[i].imshow(train_images[idx][:,:,0], cmap='gray')
        axs[i].set_title(emotions[train_labels[idx].argmax()])
        axs[i].set_xticklabels([])
        axs[i].set_yticklabels([])

def prepare_data(data):
    """ Prepare data for modeling 
        input: data frame with labels und pixel data
        output: image and label array """
    
    image_array = np.zeros(shape=(len(data), 48, 48))
    image_label = np.array(list(map(int, data['emotion'])))
    
    for i, row in enumerate(data.index):
        image = np.fromstring(data.loc[row, ' pixels'], dtype=int, sep=' ')
        image = np.reshape(image, (48, 48))
        image_array[i] = image
        
    return image_array, image_label

def plot_compare_distributions(array1, array2, title1='', title2=''):
    df_array1 = pd.DataFrame()
    df_array2 = pd.DataFrame()
    df_array1['emotion'] = array1.argmax(axis=1)
    df_array2['emotion'] = array2.argmax(axis=1)
    
    fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharey=False)
    x = emotions.values()
    
    y = df_array1['emotion'].value_counts()
    keys_missed = list(set(emotions.keys()).difference(set(y.keys())))
    for key_missed in keys_missed:
        y[key_missed] = 0
    axs[0].bar(x, y.sort_index(), color='orange')
    axs[0].set_title(title1)
    axs[0].grid()
    
    y = df_array2['emotion'].value_counts()
    keys_missed = list(set(emotions.keys()).difference(set(y.keys())))
    for key_missed in keys_missed:
        y[key_missed] = 0
    axs[1].bar(x, y.sort_index())
    axs[1].set_title(title2)
    axs[1].grid()
    
    plt.show()

In [ ]:
### Getting the shape of the dataset

print(data.shape)

# EMOTION COLUMN

In [ ]:
emotion_label_to_text = {0:'anger', 1:'disgust', 2:'fear', 3:'happiness', 4: 'sadness', 5: 'surprise', 6: 'neutral'}

In [ ]:
# Exploring the count of emotion 

data.emotion.value_counts()

In [ ]:
#  Breaking down each class of emotion 

class_weight = dict(zip(range(0, 7), (((data[data[' Usage']=='Training']['emotion'].value_counts()).sort_index())/len(data[data[' Usage']=='Training']['emotion'])).tolist()))
class_weight

# 2) Feature Engineering/Data Preprocessing for Model Data Exploration

In [ ]:
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline

from collections import Counter
emotions = {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Sad', 5: 'Surprise', 6: 'Neutral'}

train_image_array, train_image_label = prepare_data(data[data[' Usage']=='Training'])
val_image_array, val_image_label = prepare_data(data[data[' Usage']=='PrivateTest'])
test_image_array, test_image_label = prepare_data(data[data[' Usage']=='PublicTest'])


In [ ]:
train_images = train_image_array.reshape((train_image_array.shape[0], 48, 48, 1))
train_images = train_images.astype('float32')
val_images = val_image_array.reshape((val_image_array.shape[0], 48, 48, 1))
val_images = val_images.astype('float32')
test_images = test_image_array.reshape((test_image_array.shape[0], 48, 48, 1))
test_images = test_images.astype('float32')

train_labels = tf.keras.utils.to_categorical(train_image_label)
val_labels = tf.keras.utils.to_categorical(val_image_label)
test_labels = tf.keras.utils.to_categorical(test_image_label)

In [ ]:
train_images.shape

In [ ]:
val_images.shape

In [ ]:
test_images.shape

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
data[' pixels'] = data[' pixels'].apply(lambda x: np.fromstring(x, sep=' '))

# Convert the 'pixels' column to a 2D NumPy array
pixel_values = np.vstack(data[' pixels'].values)
# Flatten the 2D array to create a 1D array for each image
pixels_flat = pixel_values.reshape(pixel_values.shape[0], -1)

# Apply PCA
pca = PCA(n_components=0.98)
pca_result = pca.fit_transform(pixels_flat)

# Apply PCA to reduce dimensionality
num_components = 50  # Choose the number of components
pca = PCA(n_components=num_components)
pca_result = pca.fit_transform(pixel_values)

# Add the PCA components to the DataFrame
for i in range(num_components):
    data[f'pca_{i+1}'] = pca_result[:, i]

# Split the data into features (X) and target labels (y)
X = data.iloc[:, 3:]  # Use PCA components as features, adjust column index accordingly
y = data['emotion']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# Train a RandomForestClassifier (you can use any other classifier)
model = RandomForestClassifier(n_estimators=500, random_state=100)
model.fit(X_train, y_train)

# Predict on the testing set
y_pred = model.predict(X_test)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

# Optionally, you can visualize the data in the reduced-dimensional space
# Scatter plot of the first two principal components
plt.scatter(data['pca_1'], data['pca_2'], c=data['emotion'], cmap='viridis', alpha=0.5)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Emotion Data')
plt.show()

# Print shapes and number of components
print("Original shapes:", pixel_values.shape, pixels_flat.shape)
print("Shapes after PCA:", pca_result.shape)
print("Number of components:", pca.n_components_)


In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# Assuming 'X_training' and 'X_test' are your 4D arrays representing images
# Print the shapes of the training and test sets before PCA
print("Before PCA - Training Set Shape:", train_images.shape)
print("Before PCA - Test Set Shape:", test_images.shape)

# Reshape each image into a 1D array
xtrain_flat = train_images.reshape(train_images.shape[0], -1)
xtest_flat = test_images.reshape(test_images.shape[0], -1)

# Apply PCA
pca = PCA(n_components=0.98)
pca_train = pca.fit_transform(xtrain_flat)
pca_test = pca.transform(xtest_flat)

# Print the shapes of the transformed training and test sets
print("After PCA - Transformed Training Set Shape:", pca_train.shape)
print("After PCA - Transformed Test Set Shape:", pca_test.shape)

# Print explained variance and cumulative explained variance
print("Explained Variance by Each Principal Component:")
print(pca.explained_variance_)
print("\nExplained Variance Ratio by Each Principal Component:")
print(pca.explained_variance_ratio_)
print("\nCumulative Explained Variance:")
print(np.cumsum(pca.explained_variance_ratio_))


# 3) Data Exploration (Exploring numerical, categorical data, and plotting)

In [ ]:
# Visualizing emotion count 

plot_compare_distributions(train_labels, val_labels, title1='train labels', title2='val labels')

In [ ]:
# Visualize the data 

plot_all_emotions()

# 4) Building the model

### Model 1 - CNN

In [ ]:
#CNN MODOEL 
#defining model
#model=Sequential()
model = Sequential([
    #adding convolution layer
    #model.add(Conv2D(32,(3,3),activation=’relu’,input_shape=(28,28,1)))
    Conv2D(48, (3,3), activation = 'relu', padding = 'same', input_shape = (48,48,1)),
    Conv2D(48, (3,3), activation = 'relu', padding = 'same'),
    #adding pooling layer
    #model.add(MaxPool2D(2,2))
    MaxPool2D(2,2),
    Dropout(0.2),
    
    
    #adding fully connected layer
    #model.add(Flatten())
    #model.add(Dense(100,activation=’relu’))
    
    Flatten(),
    
    Dense(96, activation='relu'),
    Dropout(0.2),
    Dense(96, activation='relu'),
    Dropout(0.2),
    
    #adding output layer
    #model.add(Dense(10,activation=’softmax’))
    Dense(7, activation='softmax')
])

model.summary()

In [ ]:
model.compile(optimizer=Adam(lr=1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
%time 

history = model.fit(train_images, train_labels,
                    validation_data=(val_images, val_labels),
                    class_weight = class_weight,
                    epochs=10,
                    batch_size=64)

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('test accuracy:', test_acc)

### Model 1 Results

In [ ]:
loss = history.history['loss']
loss_val = history.history['val_loss']
epochs = range(1, len(loss)+1)
plt.plot(epochs, loss, 'bo', label='loss_train')
plt.plot(epochs, loss_val, 'b', label='loss_val')
plt.title('value of the loss function')
plt.xlabel('epochs')
plt.ylabel('value of the loss function')
plt.legend()*6
plt.grid()
plt.show()

In [ ]:
acc = history.history['accuracy']
acc_val = history.history['val_accuracy']
epochs = range(1, len(loss)+1)
plt.plot(epochs, acc, 'bo', label='accuracy_train')
plt.plot(epochs, acc_val, 'b', label='accuracy_val')
plt.title('accuracy')
plt.xlabel('epochs')
plt.ylabel('value of accuracy')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical
import numpy as np

In [ ]:
# Data Preprocessing
# Assuming 'pixels' column contains pixel values as a string, you need to convert it to a numpy array
data[' pixels'] = data[' pixels'].apply(lambda x: np.fromstring(x, sep=' '))


In [ ]:

# Convert labels to categorical
label_encoder = LabelEncoder()
data['emotion'] = label_encoder.fit_transform(data['emotion'])

# Split the data into training and testing sets
train_images, test_images, train_labels, test_labels = train_test_split(
    data[' pixels'].tolist(), data['emotion'].tolist(), test_size=0.2, random_state=100)

# Convert pixel values to a 2D array
X_train = np.vstack(train_images)
X_test = np.vstack(test_images)

# Convert labels to one-hot encoding
y_train = to_categorical(train_labels, num_classes=7)
y_test = to_categorical(test_labels, num_classes=7)

# Build the Neural Network model
model = Sequential()
model.add(Dense(128, input_shape=(X_train.shape[1],), activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(7, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

# Evaluate the model on the test set
predictions = model.predict(X_test)
predicted_labels = np.argmax(predictions, axis=1)
accuracy = accuracy_score(test_labels, predicted_labels)
print(f"Test Accuracy: {accuracy}")
# Create a confusion matrix
conf_mat = confusion_matrix(test_labels, predicted_labels)
print("Confusion Matrix:")
print(conf_mat)



In [ ]:
# Step 6: Evaluate the model on the testing set
y_pred = model.predict(list(test_images))  # Convert X_test to a list as well
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
# Print accuracy and other metrics
print(f'Accuracy: {accuracy}')
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

# Step 7: Make predictions on new data
# Assuming you have a new sample in variable 'new_sample'
new_sample = X_test.iloc[0].reshape(1, -1)  # Adjust this based on your actual data
predicted_emotion = model.predict(new_sample)
print(f'Predicted Emotion: {predicted_emotion[0]}')